In [0]:
df = spark.read.table("workspace.bronze.erp_px_cat_g1v2")

In [0]:
df.display()

In [0]:
## trim whitespace and change column names 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df1 = df.withColumn(field.name, trim(col(field.name)))

mapping = {
    'ID': 'product_category_code',
    'CAT': 'category',
    'SUBCAT':'sub_category',
    'MAINTENANCE':'maintenance',
}
df2 = df1.withColumnsRenamed(mapping)

In [0]:
df2.display()


In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df2.groupBy(df2.columns[0]).count().filter("count > 1")
display(duplicate_count)

### write to silver table

In [0]:
df2.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.erp_product_details_cleaned")